In [ ]:

# Experiment 8: Clustering HAR data using KMeans, DBSCAN, Hierarchical
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              calinski_harabasz_score, adjusted_rand_score,
                              normalized_mutual_info_score, confusion_matrix)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")


In [ ]:

# Load the UCI HAR dataset
# Change this path to where the UCI HAR Dataset folder is stored
data_path = "UCI HAR Dataset"

feature_names_raw = pd.read_csv(f"{data_path}/features.txt", sep="\\s+", header=None)[1].values

# Make feature names unique since some repeat (real dataset quirk)
seen = {}
feature_names = []
for name in feature_names_raw:
    if name in seen:
        seen[name] += 1
        feature_names.append(f"{name}_{seen[name]}")
    else:
        seen[name] = 0
        feature_names.append(name)

X_train = pd.read_csv(f"{data_path}/train/X_train.txt", sep="\\s+", header=None)
X_test = pd.read_csv(f"{data_path}/test/X_test.txt", sep="\\s+", header=None)

y_train = pd.read_csv(f"{data_path}/train/y_train.txt", sep="\\s+", header=None)
y_test = pd.read_csv(f"{data_path}/test/y_test.txt", sep="\\s+", header=None)

# Combine train and test for clustering (unsupervised task)
X = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
y = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

X.columns = feature_names
y.columns = ["Activity"]

activity_map = {1: "WALKING", 2: "WALKING_UPSTAIRS", 3: "WALKING_DOWNSTAIRS",
                4: "SITTING", 5: "STANDING", 6: "LAYING"}
y["ActivityName"] = y["Activity"].map(activity_map)

print(X.shape, y.shape)
X.head()


In [ ]:

# Check for missing values
print("Missing values:", X.isnull().sum().sum())

# Encode activity labels for later comparison with clusters
le = LabelEncoder()
y_encoded = le.fit_transform(y["ActivityName"])


In [ ]:

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:

# Exploratory Data Analysis
# Class distribution of activities
plt.figure(figsize=(6,4))
sns.countplot(x=y["ActivityName"])
plt.xticks(rotation=45)
plt.title("Distribution of Activities")
plt.tight_layout()
plt.show()


In [ ]:

# Dimensionality reduction with PCA for visualization
pca_vis = PCA(n_components=2, random_state=42)
X_pca_vis = pca_vis.fit_transform(X_scaled)

plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca_vis[:,0], y=X_pca_vis[:,1], hue=y["ActivityName"], palette="tab10", s=15)
plt.title("PCA Projection colored by true Activity labels")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:

# t-SNE visualization (uses a subsample for speed)
sample_idx = np.random.choice(len(X_scaled), 2000, replace=False)
X_sample = X_scaled[sample_idx]
y_sample = y["ActivityName"].values[sample_idx]

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_sample)

plt.figure(figsize=(7,5))
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y_sample, palette="tab10", s=15)
plt.title("t-SNE Projection (sample of 2000 points)")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:

# For clustering, use PCA reduced data to speed up computation and reduce noise
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print("Number of PCA components retained:", X_pca.shape[1])


In [ ]:

# Model A: K-Means Clustering
# Elbow method to choose k
wcss = []
sil_scores = []
k_values = range(2, 9)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca, labels))

elbow_df = pd.DataFrame({"k": list(k_values), "WCSS": wcss, "Silhouette": sil_scores})
elbow_df


In [ ]:

# Plot elbow curve
plt.figure(figsize=(6,4))
plt.plot(k_values, wcss, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("WCSS (Inertia)")
plt.title("Elbow Method for K-Means")
plt.tight_layout()
plt.show()

# Plot silhouette curve
plt.figure(figsize=(6,4))
plt.plot(k_values, sil_scores, marker="o", color="orange")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs k")
plt.tight_layout()
plt.show()


In [ ]:

# Choose best k based on silhouette score
best_k = k_values[int(np.argmax(sil_scores))]
print("Best k chosen:", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca)


In [ ]:

# Visualize K-Means clusters using PCA 2D projection
plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca_vis[:,0], y=X_pca_vis[:,1], hue=kmeans_labels, palette="tab10", s=15)
plt.title(f"K-Means Clusters (k={best_k})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


In [ ]:

# Model B: DBSCAN Clustering
# Try a few combinations of eps and min_samples
from sklearn.neighbors import NearestNeighbors

# Use k-distance plot to help pick eps
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_pca)
distances, indices = neighbors_fit.kneighbors(X_pca)
distances = np.sort(distances[:, -1])

plt.figure(figsize=(6,4))
plt.plot(distances)
plt.xlabel("Points sorted by distance")
plt.ylabel("5th nearest neighbor distance")
plt.title("K-distance Graph for DBSCAN eps selection")
plt.tight_layout()
plt.show()


In [ ]:

# Fit DBSCAN with chosen eps and min_samples
# Adjust eps based on the elbow seen in the k-distance graph above
dbscan = DBSCAN(eps=15, min_samples=10)
dbscan_labels = dbscan.fit_predict(X_pca)

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)
print("Estimated clusters:", n_clusters_db)
print("Noise points:", n_noise)


In [ ]:

# Visualize DBSCAN clusters
plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca_vis[:,0], y=X_pca_vis[:,1], hue=dbscan_labels, palette="tab10", s=15, legend="full")
plt.title("DBSCAN Clusters (-1 indicates noise)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


In [ ]:

# Model C: Hierarchical Agglomerative Clustering
# Use a subsample for the dendrogram since full dataset is too large to plot
sample_idx_hac = np.random.choice(len(X_pca), 300, replace=False)
X_hac_sample = X_pca[sample_idx_hac]

linked = linkage(X_hac_sample, method="ward")

plt.figure(figsize=(10,5))
dendrogram(linked, truncate_mode="lastp", p=30)
plt.title("Dendrogram (Ward linkage, sample of 300 points)")
plt.xlabel("Sample index")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()


In [ ]:

# Fit Agglomerative Clustering on full PCA data using Ward linkage
hac = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hac_labels = hac.fit_predict(X_pca)


In [ ]:

# Visualize Hierarchical clusters
plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca_vis[:,0], y=X_pca_vis[:,1], hue=hac_labels, palette="tab10", s=15)
plt.title(f"Hierarchical Clustering (Ward, k={best_k})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


In [ ]:

# Evaluation Metrics
# Internal metrics do not need true labels
def internal_metrics(X, labels, name):
    # filter out noise points for DBSCAN before scoring
    mask = labels != -1
    if len(set(labels[mask])) < 2:
        return {"Model": name, "Silhouette": np.nan, "Davies-Bouldin": np.nan, "Calinski-Harabasz": np.nan}
    return {
        "Model": name,
        "Silhouette": silhouette_score(X[mask], labels[mask]),
        "Davies-Bouldin": davies_bouldin_score(X[mask], labels[mask]),
        "Calinski-Harabasz": calinski_harabasz_score(X[mask], labels[mask])
    }

results = []
results.append(internal_metrics(X_pca, kmeans_labels, "K-Means"))
results.append(internal_metrics(X_pca, dbscan_labels, "DBSCAN"))
results.append(internal_metrics(X_pca, hac_labels, "Hierarchical"))

internal_df = pd.DataFrame(results)
internal_df


In [ ]:

# External metrics compare clusters to true activity labels
def external_metrics(true_labels, pred_labels, name):
    mask = pred_labels != -1
    return {
        "Model": name,
        "ARI": adjusted_rand_score(true_labels[mask], pred_labels[mask]),
        "NMI": normalized_mutual_info_score(true_labels[mask], pred_labels[mask])
    }

ext_results = []
ext_results.append(external_metrics(y_encoded, kmeans_labels, "K-Means"))
ext_results.append(external_metrics(y_encoded, dbscan_labels, "DBSCAN"))
ext_results.append(external_metrics(y_encoded, hac_labels, "Hierarchical"))

external_df = pd.DataFrame(ext_results)
external_df


In [ ]:

# Bar plots comparing internal metrics across algorithms
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, metric in zip(axes, ["Silhouette", "Davies-Bouldin", "Calinski-Harabasz"]):
    sns.barplot(x="Model", y=metric, data=internal_df, ax=ax)
    ax.set_title(metric)
plt.tight_layout()
plt.show()

# Bar plots comparing external metrics across algorithms
fig, axes = plt.subplots(1, 2, figsize=(10,4))
for ax, metric in zip(axes, ["ARI", "NMI"]):
    sns.barplot(x="Model", y=metric, data=external_df, ax=ax)
    ax.set_title(metric)
plt.tight_layout()
plt.show()


In [ ]:

# confusion matrix mapping K-Means clusters to true activities
cm = confusion_matrix(y_encoded, kmeans_labels)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(best_k), yticklabels=le.classes_)
plt.xlabel("Cluster")
plt.ylabel("True Activity")
plt.title("Confusion Matrix: True Activity vs K-Means Cluster")
plt.tight_layout()
plt.show()


In [ ]:

# Summary of findings
print("K-Means best k:", best_k)
print("DBSCAN clusters found:", n_clusters_db, "with", n_noise, "noise points")
print("Hierarchical clusters used:", best_k)
print()
print("Internal metrics:")
print(internal_df)
print()
print("External metrics:")
print(external_df)
